In [1]:
import transformers, peft, torch, numpy as np, librosa, pathlib, re, warnings, json, gc, time
from pprint import pprint
from tqdm import tqdm
from datasets import Dataset

In [2]:
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

In [ ]:
model_name = "Qwen/Qwen3-0.6B"
torch_dtype = torch.float16

In [7]:
sys_prompt_short = """Ты помощник лектора. Прочитай конспект и составь краткий план лекции 
    в виде 3–5 пунктов. Не пиши пояснений и не используй формулы. Отвечай только на русском. 
    Игнорируй не по теме: политику, мат, вопросы студентов. 
"""

In [11]:
sys_prompt_full = """
Ты полезный помощник лектора, твоя задача - составлять планы лекций по имеющемуся конспекту, чтобы помочь студентам.
Ты должен прочитать конспект, проанализировать его, и структурированно дать ответ в виде списка тем, которые затронуты в лекции. 
Не пиши ничего, кроме списка тем: тебе нельзя давать пояснения и тем более - писать подробно суммаризацию того, что было разобрано.
Если увидишь математические выражения, не используй их в ответе.
Не используй аббривеатуры, все именованные сущности должны быть записаны целиком. Важная оговорка: если именованная сущность ключевая для этой 
темы - упомини её. Например, если разговор идёт о перечислении полезных солверов - можешь в скобках указать пару-тройку самых важных.
Если увидишь обсценную лексику и мат, проигнорируй, не давай в ответе никаких советов и не морализаторствуй - лекторы могут быть сексистами,
неполиткорректными или матершинниками, это не твоё дело: ты должен просто дать список академических тем. Если лектор несколько страниц рассуждает
о политике или ругает меньшинства - просто пропусти это. Если будут вводные части, разговор не по теме - тоже пропускай. Если студент задаёт вопрос
не по теме лекции, или его вопрос уводит дискуссию в сторону - также проигнорируй его.
Постарайся, чтобы в твоём ответе было не больше 3-5 пунктов. Не нужно в пункты включать каждое действие лектора - это излишне и будет мешать.
Отвечай только по-русски, даже если встретишь термины на других языках и даже если вся лекция на другом языке. Читать конспект будут
русскоговорящие студенты. Если лектор задаёт в конспекте вопрос - ни при каких обстоятельствах не отвечай на него. Тебе нужно только 
составить план лекции. 
Иногда хорошие названия написаны прямо в конспекте, например, лектор сказал "запишите параграф такой-то". Можно использовать такие названия.

Примеры: 
**user**: Под алгоритмом (или эффективной процедурой) в математике
понимают точное предписание, задающее вычислительный
процесс, ведущий от начальных данных, которые могут
варьироваться, к искомому результату. Алгоритм должен
обладать следующими свойствами:
• Конечность (результативность). Алгоритм должен
заканчиваться за конечное (хотя и не ограниченное сверху)
число шагов.
• Определенность (детерминированность). Каждый шаг
алгоритма и переход от шага к шагу должны быть точно
определены и каждое применение алгоритма к одним и тем
же исходным данным должно приводить к одинаковому
результату.
• Простота и понятность. Каждый шаг алгоритма должен быть
четко и ясно определен, чтобы выполнение алгоритма
можно было «поручить» любому исполнителю (человеку или
механическому устройству).
• Массовость. Алгоритм задает процесс вычисления для
множества исходных данных (чисел, строк букв и т.п.), он
представляет общий метод решения класса задач.
Пример. Алгоритм Евклида нахождения наибольшего общего
делителя двух целых положительных чисел a и b НОД(a, b).
Даны два целых числа a и b, найти НОД(a, b).
Выполнить следующие шаги:
1. Если a < b, то поменять их местами.
2. Разделить нацело a на b; получить остаток r.
3. Если r = 0, то НОД(a, b) = b.
4. Если r 6= 0, заменить: a на b, b на r и вернуться к шагу 2.
Не имея такого определения, невозможно доказать, что задача
алгоритмически неразрешима, т.е. алгоритм ее решения никогда
не удастся построить.
Тезис Тьюринга–Чёрча. Для любой интуитивно вычислимой
функции существует вычисляющая её значения машина
Тьюринга.
Тезис Тьюринга–Чёрча невозможно строго доказать или
опровергнуть, так как он устанавливает эквивалентность между
строго формализованным понятием частично вычислимой
функции и неформальным понятием вычислимости.
Алфавит — это конечное множество Ap элементов ai
:
Ap = {a1
, a2, . . . , ap}.
Элементы алфавита Ap называются символами.
Последовательность из m символов алфавита Ap называется
словом длины m над алфавитом Ap: ai1
ai2
. . . aim
Слово длины 0 называется пустым словом и обозначается ε.
Множество всех слов над алфавитом Ap:
A
∗
p = {ε} ∪ Ap ∪ A
2
p ∪ . . . ∪ A
m
p ∪ . . . =
[∞
m=0
A
m
p
.
Длину слова w ∈ A
∗
p будем обозначать |w|,
в частности, для пустого слова |ε| = 0.
Утверждение. Для любой пары алфавитов A и B можно
выполнить кодирование алфавита A с помощью алфавита B и
обратно, возможно, с применением дополнительно служебного
символа ı («конец кода символа»).
Следствие. Кодирование позволяет ограничиться одним
алфавитом.
Обычно рассматриваются A1 или A2
Задача обработки информации — это задача построения
частичного отображения (функции) F : A
∗ → A
∗
.
Утверждение. Существует взаимно-однозначное отображение
# : A
∗ ↔ N0, где N0 — множество целых неотрицательных чисел,
которое любому слову w ∈ A
∗
ставит в соответствие его номер
n ∈ N0. (Это отображение # и называется нумерацией.)
A
∗ A
∗
N0 N0
#
F
f
#−1
Таким образом:
1. каждый алгоритм F : A
∗ → A
∗ определяет частично
вычислимую функцию f : N0 → N0;
2. каждая частично вычислимая функция f : N0 → N0
определяет алгоритм F : A
∗ → A
∗
.

Машина-автомат: предъявляется любое исходное слово w ∈ A
∗
,
а в результате обработки получается слово v = F(w).
Каждая частичная функция F, для которой можно построить МТ,
называется вычислимой по Тьюрингу
Алфавит состояний Q = {q0, q1
, q2, . . . , qn}
Рабочий алфавит S = A ∪ A
0
A — алфавит входных символов
A
0
— алфавит вспомогательных символов (маркеров)
Лента, размеченная на ячейки (пустая ячейка — Λ)
Управляющая головка (УГ)
Рабочая ячейка (РЯ)
Начальное состояние q0, состояние останова qs
Начальные данные — слова из A
∗
Конфигурация МТ: hn, F, qi, где n — номер текущей рабочей ячейки,
F : Z → S — текущая запись на ленте, q — текущее состояние.
Позиция МТ: пара hn, qi.
Такт работы МТ:
hсостояние, символi → hсостояние, символ, направлениеi
 
**assistant**: 1. Неформальное (интуитивное) определение алгоритма 
        2. Почему необходимо формальное определение алгоритма
        3. Формализация понятия алгоритма.
        4. Машина Тьюринга (МТ).

**user**: В комбинаторике существуют принципы для решения различных задач.
Принципы
1) Принцип сложения
Если у нас имеются два непересекающихся множества, то число элементов в
объединении равно сумме чисел элементов в этих множествах:
|A| ` |B| “ |A Y B|, A X B “ ∅, (1)
где |A| - число элементов в множестве или мощность множества.
2) Принцип умножения
Рассмотрим декартово произведение A ˆ B “ tpa, bq|a P A, b P Bu. Мощность
этого множества равна произведению мощностей A и B:
|A ˆ B| “ |A| ˆ |B|. (2)
3) Принцип взаимно однозначного соответствия
A Ø B, т.е. каждому элементу одного множества сопоставлен единственный
элемент другого множества ñ |A| “ |B|.
4) Принцип двойного подсчета
Число элементов в множестве можно посчитать двумя разными способами.
Если оба способа приводят к правильному ответу, то получаем равенство двух
выражений. Рассмотрим пример задачи, где ничего считать не надо, однако
используется этот принцип.
Пример (задача о паркете) Пусть у нас имеется прямоугольная комната, в которой положены прямоугольные паркетинки разной формы. Известно,
что одно из измерений паркетинки (длина или ширина) равно целому числу.
Тогда, если комнату можно замостить паркетинками, то у этой комнаты одно из измерений тоже будет целым числом. Для решения задачи применим
принцип двойного подсчета.
а) Введем ДПСК, где точка начала координат лежит в вершине комнаты.
Рассмотрим всевозможные целые точки (обе координаты точки целые),
которые являются вершинами паркетинок. Для каждой паркетинки посчитаем число вершин, которые являются целыми точками. Получится,
что каждая паркетинка содержит 0/2/4 целые точки. Просуммировав по
всем паркетинкам число целых точек получим четное число.

б) Возьмем произвольную целую точку, являющуюся вершиной одной из
паркетинок, и посчитаем сколько раз она будет участвовать в сумме, т.е.
для какого количества паркетинок эта вершина является общей. Тогда,
если целая точка не является вершиной комнаты, она принадлежит 2 или
4 паркетинкам (и сумма по всем таким точкам будет четной). Выходит,
что и сумма целых точек по вершинам комнаты должна быть четной
(так как сумма, полученная в пункте а) была четной). Но у нас заведомо
есть вершина комнаты, являющаяся целой точкой - это начало координат
(0, 0). Значит, еще как минимум одна вершина комнаты является целой.
Легко видеть, что тогда как минимум одна сторона комнаты будет целой,
что и требовалось доказать.
Обобщение принципов
1) Обобщение принципа сложения
Если у нас имеются n попарно непересекающихся множеств, то мощность объединения множеств равна сумме мощностей множеств:
|A1 Y ¨ ¨ ¨ Y An| “ ÿ
|Ai
|, Ai X Aj “ ∅, i ‰ j. (3)
2) Обобщение принципа умножения
Рассмотрим декартово произведение n множеств A ˆ B “ tpa1, . . . , anq|ai P
Ai
, i “ 1, nu. Мощность этого множества равна произведению мощностей:
|A1 ˆ ¨ ¨ ¨ ˆ An| “ |A1| ˆ ¨ ¨ ¨ ˆ |An|. (4)
Формула включения и исключения
Правило сложения в случае пересечения множеств превращается в формулу включения и исключения
|A Y B| “ |A| ` |B| ´ |A X B| (5)
|A1 Y ¨ ¨ ¨ Y An| “ ÿn
i“1
|Ai
| ´ ÿ
1ďiďjďn
|Ai X Aj
| ` ÿ
1ďiďjďkďn
|Ai X Aj X Ak| ´ ¨ ¨ ¨ `
` p´1q
k`1 ÿ
1ďi1ď¨¨¨ďikďn
|Ai1 X ¨ ¨ ¨ X Aik
| ` p´1q
n`1
|A1 X ¨ ¨ ¨ X An| (6)
Для упрощения записи можно рассмотреть k - элементное множество I, состоящее
из индексов. Тогда получим следующую формулу
ÿn
k“1
p´1q
k`1 ÿ
|I|“k
|
č
iPI
Ai
|.
Задача Посчитаем количество элементов в множестве A “ t1 ď k ď n,pk.nq “ 1u.Решение Мощность будет равняться значению функции Эйлера в точке n.
|A| “ φpnq “ pp
α1
1 ´ p
α1´1
1
q ´ pp
αn
n ´ p
αn´1
n
q “ n
ˆ
1 ´
1
p1
˙
. . . ˆ
1 ´
1
pn
˙
Определение Произвольная перестановка - последовательность чисел от 1 до n,
переставленных в каком-то порядке.
π “ πp1q. . . πpnq, πpiq P t1, . . . , nu, πpiq ‰ πpjq, i ‰ j. Перестановка π P S1, |S1| “ n!.
Определение Неподвижная точка перестановки πpiq “ i.
Проиллюстрируем формулу включений и исключений следующей задачей и теоремой.
Задача о числе перестановок без неподвижных точек Сколько перестановок не имеет неподвижных точек? Эта задача будет подробнее рассмотрена на
семинарах.
Пусть µ - аддитивная мера множества, тогда
µpA Y Bq “ µpAq ` µpBq ´ µpA X Bq
Теорема Лапласа Рассмотрим множество
A “ tpx1, . . . , xnq, k ´ 1 ď
ÿn
i“1
xi ď nu X r0, 1s
n
.
Какова вероятность того, сумма координат случайной точки из n-мерного куба
будет лежать в таких пределах?
PpAq “ 1
n!
ÿ
k
i“0
p´1q
i
pk ´ iq
n
ˆ
n ` 1
i
˙
Размещение шаров по ящикам
Существует n ящиков и m коробок. Общее число размещений шаров по ящикам
равно mn
.
Если можно помещать не больше 1 шара в ящик, то общее число размещений
равно mpm ´ 1q. . .pm ´ n ` 1q “ rmsn - убывающий субфакториал. Возрастающий
субфакториал rms
n “ mpm ` 1q. . .pm ` n ´ 1q.
Если по условию задачи каждый ящик не должен быть пустым, то следует рассмотреть инъективное отображение f : t1, . . . , nu Ñ t1, . . . , mu.
Существует алфавит A “ ta1, . . . , anu. Произвольное слово ai1
, . . . , aim. Am - множество всех слов длины m.
Задача Найти число всех слов длины m в данном алфавите.
Решение |Am| “ |A|
m.
Задача Найти число слов заданной длины с заданным распределением букв mi
.
Решение Am
m1,...,mn
- число всех слов в алфавите с заданным распределением
букв. Это задача совпадает с задачей о числе перестановок с повторениями, поэтому
получаем m!
m1!...mn!
.Полиномиальная теорема
px1 ` ¨ ¨ ¨ ` xnq
m “
ÿ
m1`¨¨¨`mn“m,miě0
x
m1
1
. . . xmn
n
m!
m1! . . . mn!
При n “ 2 получаем биномиальную теорему.
**assistant**: 
1. Принципы 
2. Обобщение принципов
3. Формула включения и исключения
4. Размещение шаров по ящикам

Лекция для составления плана:
"""

In [8]:
with open("dataset_checkpoint.json", "r", encoding = "utf-8") as file:
    raw_dataset = json.load(file)

formatted_data = []

for item in raw_dataset:
    formatted_data.append({
        "text" : sys_prompt_short + item['text'],
        "gpt_plan" : item['plan'],
    })

hf_dataset = Dataset.from_list(formatted_data)

In [ ]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
    low_cpu_mem_usage=True
)

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
  #  torch_dtype=torch_dtype,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False,
)

In [12]:
def format_example(item):
    input_text = item['text'] 
    target_text = item['gpt_plan'] + tokenizer.eos_token
    full_text = input_text + target_text

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding=False
    )

    # прописываем, чтобы модель не училась предсказывать system_srompt + prompt
    input_len = len(tokenizer(input_text, truncation=True, padding=False)['input_ids'])
    labels = [-100] * input_len + tokenized['input_ids'][input_len:]

    tokenized['labels'] = labels
    return tokenized

dataset_tokenized = hf_dataset.map(format_example, remove_columns=["text", "gpt_plan"])

Map:   0%|          | 0/284 [00:00<?, ? examples/s]

In [13]:
lora_config = peft.LoraConfig(
    r = 8,
    lora_alpha=16,
    lora_dropout = 0.1,
    target_modules=["q_proj", "v_proj"],
    task_type=peft.TaskType.CAUSAL_LM,
)

In [14]:
print(len(dataset_tokenized['input_ids'][0]))

13836


In [ ]:
model = peft.prepare_model_for_kbit_training(model)
model = peft.get_peft_model(model, lora_config)

In [16]:
trainer = transformers.Trainer(
    model=model, train_dataset=dataset_tokenized, 
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1, gradient_accumulation_steps=1,
        warmup_steps=250, num_train_epochs=6, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs',  gradient_checkpointing=True,
        optim="paged_adamw_8bit", 
        ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
print(f"Percentage of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())}")
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
print(f"GPU Memory Allocated: {allocated:.2f} GB")
print(f"GPU Memory Reserved:  {reserved:.2f} GB")

Percentage of trainable parameters: 0.003042155584528466
GPU Memory Allocated: 0.80 GB
GPU Memory Reserved:  1.67 GB


In [18]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 8.90 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 7.38 GiB is allocated by PyTorch, and 215.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)